# 05 - End-to-end pipeline audit (CPU)

**Required Kaggle settings:** Accelerator **None**. Internet **On** for the clone in cell 2,
or **Off** if you mount the dataset produced by notebook 01 instead.

This notebook is the one to run before any deployment review. It exercises the full
L1 -> L2 -> L3 -> L4 path with stub perception adapters, then audits the two properties that
actually determine whether caregivers will keep using the system:

1. **Every alert is fully explainable** - all six fields present, no clinical language.
2. **The false-alarm budget holds** - fall thresholds are chosen per resident-week, not by AUC.

No GPU and no camera are needed. That is deliberate: a test suite that requires hardware is a
test suite that stops being run.


In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/EmotionSense-Extended')
BUNDLED = Path('/kaggle/input')

bundle = next(BUNDLED.glob('*/code/src'), None) if BUNDLED.exists() else None
if bundle is not None:
    sys.path.insert(0, str(bundle))
    print('using bundled source from', bundle)
else:
    if not REPO_DIR.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PARTHG0106/EmotionSense-Extended.git', str(REPO_DIR)],
            check=True,
        )
    sys.path.insert(0, str(REPO_DIR / 'src'))
    print('using cloned source from', REPO_DIR / 'src')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydantic>=2', 'pyyaml'], check=True)


In [ ]:
# Run the committed test suite. If this fails, stop: the notebook results below would be
# measuring broken logic.
if REPO_DIR.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=str(REPO_DIR), text=True,
    )
    print('exit code:', result.returncode)
else:
    print('tests are not bundled into the dataset; skipping (run them in CI or locally)')


## A synthetic day, including one fall

The trace below is built from geometry, not from a model: an upright bounding box that drops,
goes horizontal and stays still. That is exactly what the deployed rule-first detector keys
on, so this reproduces the real detection path rather than a mocked confidence score.


In [ ]:
from datetime import datetime, timedelta

from wellbeing.config import load_config
from wellbeing.contracts.common import BBox, SubjectKind, TimeWindow
from wellbeing.contracts.perception import Posture
from wellbeing.pipeline import Pipeline

config_path = next(
    (p for p in [
        (REPO_DIR / 'configs' / 'default.yaml') if REPO_DIR.exists() else None,
        next(Path('/kaggle/input').glob('*/code/configs/default.yaml'), None),
    ] if p is not None and p.exists()),
    None,
)
config = load_config(config_path) if config_path else load_config()
print('config loaded from', config_path)
print('fall drop ratio threshold:', config.activity.fall.centroid_drop_ratio)
print('identity floor for behavioural claims:', config.reasoning.identity_confidence_floor)


In [ ]:
T0 = datetime(2026, 3, 20, 14, 0, 0)
UPRIGHT = BBox(x1=100.0, y1=80.0, x2=180.0, y2=300.0)
FLOOR = BBox(x1=100.0, y1=250.0, x2=320.0, y2=330.0)

# The pipeline's own test helper builds contract-valid frames; reuse it so this notebook
# cannot drift from the tested construction path.
if REPO_DIR.exists():
    sys.path.insert(0, str(REPO_DIR))
from tests.conftest import make_frame  # noqa: E402

frames = [make_frame(i, Posture.STANDING, bbox=UPRIGHT, speed=25.0) for i in range(4)]
frames += [make_frame(i, Posture.LYING, bbox=FLOOR, speed=1.0, seconds=1.0) for i in range(5, 40)]

pipeline = Pipeline(config)
events, alerts = pipeline.process_frames(frames)
print(f'{len(events)} events, {len(alerts)} alerts')
for event in events:
    print(f'  {event.label.value:<24} {event.duration_seconds:6.1f}s  conf={event.confidence:.2f}  via {event.source.value}')


## Explanation audit

The target is 100% six-field completeness. Anything less means a caregiver would see an alert
they cannot interrogate, and the reliable response to that is to start ignoring alerts.


In [ ]:
BANNED = config.reasoning.banned_phrases
VAGUE = ['unusual behavior', 'unusual activity', 'anomaly detected', 'something is wrong']

failures = []
for alert in alerts:
    print('=' * 78)
    print(f'[{alert.severity.value.upper()}] {alert.kind.value}   confidence={alert.confidence:.2f}')
    for line in alert.explanation.as_lines():
        print('  ' + line)
    print(f'  human review required: {alert.requires_human_review}')
    if alert.suppressed_reason:
        print(f'  SUPPRESSED: {alert.suppressed_reason}')

    if not alert.explanation.is_complete:
        failures.append((alert.alert_id, 'incomplete', alert.explanation.missing_fields))
    text = ' '.join(alert.explanation.as_lines()).lower()
    for phrase in list(BANNED) + VAGUE:
        if phrase.lower() in text:
            failures.append((alert.alert_id, 'banned/vague phrase', phrase))

print('=' * 78)
assert not failures, failures
print(f'explanation completeness: {len(alerts)}/{len(alerts)} alerts pass')


## False-alarm budget sweep

Replace the synthetic scores with your held-out fall scores and the true negative duration of
the evaluation footage. The number that matters is the third column: a detector above about
one false alarm per resident-week gets muted, and a muted detector has zero recall.


In [ ]:
import random

train_dir = next(
    (p for p in [
        (REPO_DIR / 'kaggle' / 'train') if REPO_DIR.exists() else None,
        next(Path('/kaggle/input').glob('*/code/train'), None),
    ] if p is not None and p.exists()),
    None,
)
sys.path.insert(0, str(train_dir))
from train_fall import MAX_FALSE_ALARMS_PER_RESIDENT_WEEK, threshold_for_budget  # noqa: E402

rng = random.Random(42)
scores = [(min(1.0, max(0.0, rng.gauss(0.82, 0.10))), True) for _ in range(60)]
scores += [(min(1.0, max(0.0, rng.gauss(0.18, 0.12))), False) for _ in range(20000)]
NEGATIVE_HOURS = 24 * 7 * 8  # eight resident-weeks of ordinary footage

threshold, recall, false_per_week = threshold_for_budget(scores, NEGATIVE_HOURS)
print(f'chosen threshold      : {threshold:.3f}')
print(f'recall at threshold   : {recall:.3f}  (target >= 0.95)')
print(f'false alarms/res-week : {false_per_week:.2f}  (budget <= {MAX_FALSE_ALARMS_PER_RESIDENT_WEEK})')
print()
print('For contrast, the naive 0.5 threshold:')
weeks = NEGATIVE_HOURS / (24 * 7)
naive_recall = sum(1 for s, y in scores if y and s >= 0.5) / sum(1 for _, y in scores if y)
naive_false = sum(1 for s, y in scores if not y and s >= 0.5) / weeks
print(f'  recall {naive_recall:.3f}, false alarms/resident-week {naive_false:.2f}')


## Deployment review gates

| Gate | Target |
| --- | --- |
| Explanation completeness | 100% of alerts, all six fields |
| Fall recall | >= 0.95 at <= 1 false alarm / resident-week, <= 10s latency |
| Identity | HOTA >= 60, < 2 ID switches / hour / resident, false-merge <= 0.1% |
| ADL | macro-F1 >= 0.70 |
| Anomaly | AUC >= 0.80, precision@10 >= 0.6 |
| Cold start | zero deviation alerts during the 14-day baseline warmup |

If a gate fails, the correct response is to narrow scope rather than ship a noisier system.
A monitor that caregivers mute is strictly worse than no monitor, because it creates the
appearance of coverage that nobody is actually providing.
